# 第7章：構建聊天應用程式
## OpenAI API 快速入門

此筆記本改編自[Azure OpenAI 範例存放庫](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst)，其中包含存取[Azure OpenAI](notebook-azure-openai.ipynb) 服務的筆記本。

Python OpenAI API 也可用於 Azure OpenAI 模型，僅需少量修改。詳情請參閱：[如何使用 Python 在 OpenAI 和 Azure OpenAI 端點之間切換](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# 概覽  
「大型語言模型係將文字映射到文字嘅函數。給定一串輸入文字，大型語言模型會嘗試預測接下來嘅文字內容」（1）。呢個「快速入門」筆記本將會向用戶介紹高階嘅LLM概念、開始使用AML所需嘅核心套件、對提示設計嘅簡介，同埋多個不同使用案例嘅短小例子。 


## 目錄  

[概覽](#overview)  
[如何使用 OpenAI 服務](#how-to-use-openai-service)  
[1. 建立你的 OpenAI 服務](#1.-creating-your-openai-service)  
[2. 安裝](#2.-installation)    
[3. 憑證](#3.-credentials)  

[使用案例](#use-cases)    
[1. 文本摘要](#1.-summarize-text)  
[2. 文本分類](#2.-classify-text)  
[3. 生成新產品名稱](#3.-generate-new-product-names)  
[4. 微調分類器](#4.fine-tune-a-classifier)  

[參考](#references)


### 建立你的第一個提示  
這個簡短的練習將提供一個基本介紹，教你如何向 OpenAI 模型提交提示，以完成簡單的「摘要」任務。


<strong>步驟</strong>:  
1. 在你的 Python 環境中安裝 OpenAI 函式庫  
2. 載入標準輔助函式庫，並設定你為所建立的 OpenAI 服務所使用的典型 OpenAI 安全憑證  
3. 為你的任務選擇一個模型  
4. 為模型建立一個簡單的提示  
5. 將請求提交到模型 API！


### 1. 安裝 OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. 導入輔助庫並實例化憑證


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. 尋找合適的模型  
像 GPT-4o 和 GPT-4o mini 這類模型能理解並產生自然語言，並且可在 OpenAI 平台上使用，提供不同的運算能力和速度，適合不同的任務需求。


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. 提示設計  

「大型語言模型的神奇之處在於，通過訓練使這種預測誤差在大量文本中最小化，模型最終學會了對這些預測有用的概念。例如，它們學會了像」(1):

* 如何拼寫
* 語法如何運作
* 如何改寫
* 如何回答問題
* 如何進行對話
* 如何用多種語言寫作
* 如何編碼
* 等等。

#### 如何控制大型語言模型  
「在所有輸入大型語言模型的資料中，最具影響力的絕大多數是文本提示」(1)。

大型語言模型可以透過幾種方式提示來產生輸出：

指示：告訴模型你想要什麼
補全：誘導模型補全你想要的開頭內容
示範：向模型展示你想要的內容，方式為：
在提示中提供幾個範例
在微調訓練資料集中提供數百或數千個範例」



#### 建立提示的三個基本指引：

**展示並說明。** 透過指示、範例，或兩者結合，讓模型清楚知道你想要什麼。如果你想讓模型將項目列表按字母順序排列，或按照情感分類一段文字，請展示給它看這就是你想要的。

**提供品質良好的資料。** 如果你想建立分類器或讓模型遵循某個模式，請確保有足夠的範例。務必校對你的範例——模型通常足夠聰明能看穿基本拼寫錯誤並給出回應，但它可能會認為這是故意的，而影響回應結果。

**檢查你的設定。** 溫度和 top_p 設定控制模型生成回應時的決定性。如果你要求回應只有一個正確答案，你就會想把這些設置調低；如果你想要更多元的回應，則可以調高它們。人們在這些設定上犯的最常見錯誤是以為它們是控制「聰明程度」或「創意」。


來源: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. 提交！


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### 重複相同的呼叫，結果有何比較？ 


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## 摘要文本  
#### 挑戰  
透過在文本段落末尾加入「tl;dr:」來總結文本。注意模型如何在沒有額外指令的情況下理解並執行多項任務。你可以嘗試使用比 tl;dr 更具描述性的提示詞來修改模型的行為，並自訂你所獲得的摘要(3)。  

近來的研究顯示，在大量文本語料上進行預訓練，接著在特定任務上做微調，可以在許多自然語言處理任務和基準測試上取得顯著提升。儘管架構通常對任務無差異，但此方法仍需針對任務特定的數千或數萬示例進行微調。相較之下，人類通常只需要少量示例或簡單指示就能執行新的語言任務——這是當前的自然語言處理系統仍大部分難以做到的。本文展示了擴大量語言模型規模如何大幅提升無特定任務需求的少量示例表現，有時甚至能與過去最先進的微調方法競爭。  



摘要  


# 多種使用案例的練習  
1. 摘要文本  
2. 分類文本  
3. 產生新產品名稱


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## 文字分類  
#### 挑戰  
將項目分類至推理時提供的類別。在以下範例中，我們在提示詞中同時提供了類別及要分類的文字(*playground_reference)。 

客戶查詢：您好，我筆記型電腦鍵盤上的一個按鍵最近壞了，我需要更換一個：  

分類類別：  


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## 產生新產品名稱
#### 挑戰
從範例詞彙中建立產品名稱。我們在提示中包含了關於我們將要為其產生名稱的產品資訊。我們同時提供了類似的範例來展示我們希望收到的模式。我們也將溫度值設得很高，以增加隨機性和更具創意的回應。

產品描述：家用奶昔機
種子詞：快速、健康、緊湊。
產品名稱：HomeShaker、Fit Shaker、QuickShake、Shake Maker

產品描述：一雙可適合任何腳大小的鞋子。
種子詞：適應性強、合腳、全能合腳。


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# 參考資料  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [微調 GPT 模型以分類文本的最佳實踐](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# 尋求更多協助  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# 貢獻者
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
